In [ ]:
!pip install -q datasets tiktoken

import os
import math
import time
import inspect
from dataclasses import dataclass
import torch
import torch.nn as nn
from torch.nn import functional as F
import numpy as np
from tqdm.auto import tqdm
from datasets import load_dataset
import tiktoken

In [ ]:
if not os.path.exists("train.bin"):
    print("Downloading and processing dataset...")
    ds = load_dataset("roneneldan/TinyStories", split="train")
    enc = tiktoken.get_encoding("gpt2")

    def process(example):
        ids = enc.encode_ordinary(example['text'])
        ids.append(enc.eot_token) # Add End of Text token implies end of story
        return {'ids': ids, 'len': len(ids)}

    tokenized = ds.map(
        process,
        remove_columns=['text'],
        desc="Tokenizing",
        num_proc=os.cpu_count(), # Use all CPU cores
    )

    for split, dset in [('train', tokenized), ('val', tokenized)]: # Using same for simple demo, ideally split
        arr_len = np.sum(dset['len'], dtype=np.uint64)
        filename = f'{split}.bin'
        dtype = np.uint16
        arr = np.memmap(filename, dtype=dtype, mode='w+', shape=(arr_len,))

        idx = 0
        # Batch write for speed
        total_batches = 100
        for batch_idx in tqdm(range(total_batches), desc=f'Writing {filename}'):
            batch = dset.shard(num_shards=total_batches, index=batch_idx, contiguous=True).with_format('numpy')
            arr_batch = np.concatenate(batch['ids'])
            arr[idx : idx + len(arr_batch)] = arr_batch
            idx += len(arr_batch)
        arr.flush()
    print("Data preparation complete.")
else:
    print("Data already exists. Skipping processing.")
    enc = tiktoken.get_encoding("gpt2")

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00004-2d5a1467fff108(…):   0%|          | 0.00/249M [00:00<?, ?B/s]

data/train-00001-of-00004-5852b56a2bd28f(…):   0%|          | 0.00/248M [00:00<?, ?B/s]

data/train-00002-of-00004-a26307300439e9(…):   0%|          | 0.00/246M [00:00<?, ?B/s]

...

data/validation-00000-of-00001-869c898b5(…):   0%|          | 0.00/9.99M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/2119719 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/21990 [00:00<?, ? examples/s]

Tokenizing (num_proc=2):   0%|          | 0/2119719 [00:00<?, ? examples/s]

TimeoutError: 

In [ ]:
class LayerNorm(nn.Module):
    def __init__(self, ndim, bias):
        super().__init__()
        self.weight = nn.Parameter(torch.ones(ndim))
        self.bias = nn.Parameter(torch.zeros(ndim)) if bias else None
    def forward(self, x):
        return F.layer_norm(x, self.weight.shape, self.weight, self.bias, 1e-5)

class CausalSelfAttention(nn.Module):
    def __init__(self, config):
        super().__init__()
        assert config.n_embd % config.n_head == 0
        self.c_attn = nn.Linear(config.n_embd, 3 * config.n_embd, bias=config.bias)
        self.c_proj = nn.Linear(config.n_embd, config.n_embd, bias=config.bias)
        self.attn_dropout = nn.Dropout(config.dropout)
        self.resid_dropout = nn.Dropout(config.dropout)
        self.n_head = config.n_head
        self.n_embd = config.n_embd
        self.dropout = config.dropout
        # Flash Attention is supported on T4 (Float16)
        self.flash = hasattr(torch.nn.functional, 'scaled_dot_product_attention')

    def forward(self, x):
        B, T, C = x.size()
        q, k, v = self.c_attn(x).split(self.n_embd, dim=2)
        k = k.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        q = q.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)
        v = v.view(B, T, self.n_head, C // self.n_head).transpose(1, 2)

        if self.flash:
            # Efficient attention using Flash Attention 2 (if available) or 1
            y = F.scaled_dot_product_attention(q, k, v, attn_mask=None, dropout_p=self.dropout if self.training else 0, is_causal=True)
        else:
            # Fallback for older PyTorch versions
            att = (q @ k.transpose(-2, -1)) * (1.0 / math.sqrt(k.size(-1)))
            att = att.masked_fill(self.bias[:, :, :T, :T] == 0, float('-inf'))
            att = F.softmax(att, dim=-1)
            att = self.attn_dropout(att)
            y = att @ v

        y = y.transpose(1, 2).contiguous().view(B, T, C)
        y = self.resid_dropout(self.c_proj(y))
        return y

class MLP(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.c_fc = nn.Linear(config.n_embd, 4 * config.n_embd, bias=config.bias)
        self.gelu = nn.GELU()
        self.c_proj = nn.Linear(4 * config.n_embd, config.n_embd, bias=config.bias)
        self.dropout = nn.Dropout(config.dropout)
    def forward(self, x):
        return self.dropout(self.c_proj(self.gelu(self.c_fc(x))))

class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.ln1 = LayerNorm(config.n_embd, config.bias)
        self.attn = CausalSelfAttention(config)
        self.ln2 = LayerNorm(config.n_embd, config.bias)
        self.mlp = MLP(config)
    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.mlp(self.ln2(x))
        return x

@dataclass
class GPTConfig:
    block_size: int = 1024
    vocab_size: int = 50304 # GPT-2 vocab_size of 50257, padded up to nearest multiple of 64 for efficiency
    n_layer: int = 12
    n_head: int = 12
    n_embd: int = 768
    dropout: float = 0.0
    bias: bool = True

class GPT(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.config = config
        self.transformer = nn.ModuleDict(dict(
            wte = nn.Embedding(config.vocab_size, config.n_embd),
            wpe = nn.Embedding(config.block_size, config.n_embd),
            drop = nn.Dropout(config.dropout),
            h = nn.ModuleList([Block(config) for _ in range(config.n_layer)]),
            ln_f = LayerNorm(config.n_embd, config.bias),
        ))
        self.lm_head = nn.Linear(config.n_embd, config.vocab_size, bias=False)
        self.transformer.wte.weight = self.lm_head.weight # Weight tying

        # Init weights
        self.apply(self._init_weights)
        for pn, p in self.named_parameters():
            if pn.endswith('c_proj.weight'):
                torch.nn.init.normal_(p, mean=0.0, std=0.02/math.sqrt(2 * config.n_layer))

    def _init_weights(self, module):
        if isinstance(module, nn.Linear):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if module.bias is not None:
                torch.nn.init.zeros_(module.bias)
        elif isinstance(module, nn.Embedding):
            torch.nn.init.normal_(module.weight, mean=0.0, std=0.02)

    def forward(self, idx, targets=None):
        device = idx.device
        b, t = idx.size()
        pos = torch.arange(0, t, dtype=torch.long, device=device)
        tok_emb = self.transformer.wte(idx)
        pos_emb = self.transformer.wpe(pos)
        x = self.transformer.drop(tok_emb + pos_emb)
        for block in self.transformer.h:
            x = block(x)
        x = self.transformer.ln_f(x)
        if targets is not None:
            logits = self.lm_head(x)
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1), ignore_index=-1)
        else:
            logits = self.lm_head(x[:, [-1], :])
            loss = None
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_k=None):
        for _ in range(max_new_tokens):
            idx_cond = idx if idx.size(1) <= self.config.block_size else idx[:, -self.config.block_size:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / temperature
            if top_k is not None:
                v, _ = torch.topk(logits, min(top_k, logits.size(-1)))
                logits[logits < v[:, [-1]]] = -float('Inf')
            probs = F.softmax(logits, dim=-1)
            idx_next = torch.multinomial(probs, num_samples=1)
            idx = torch.cat((idx, idx_next), dim=1)
        return idx

In [ ]:
config = GPTConfig(
    vocab_size=50257,
    block_size=256,   # Context window
    n_layer=8,        # Increased layers
    n_head=8,         # Increased heads
    n_embd=512,       # Increased embedding size
    dropout=0.0,
    bias=False        # False usually better for modern LLMs
)

model = GPT(config)

print("-" * 60)
print(f"MODEL PARAMETER BREAKDOWN (Total Goal: ~50M)")
print("-" * 60)
total_params = 0
for name, p in model.named_parameters():
    # Only printing main blocks to keep output clean
    if ".0." in name or "wte" in name or "lm_head" in name:
        print(f"{name:20s} | Shape: {str(p.shape):15s} | Params: {p.numel()}")
    total_params += p.numel()
print("-" * 60)
print(f"TOTAL PARAMETERS: {total_params/1e6:.2f} Million")
print("-" * 60)

In [ ]:
max_iters = 5000       # Targeted for 1 hour on T4
eval_interval = 500
warmup_iters = 100
learning_rate = 6e-4   # Higher LR for smaller model/faster convergence
batch_size = 64        # Maximizing T4 memory (16GB)
gradient_accumulation_steps = 4

device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Training on: {device}")

model.to(device)
# Compile the model (HUGE speedup on modern PyTorch)
print("Compiling model... (this takes a minute but speeds up training)")
model = torch.compile(model)

optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, betas=(0.9, 0.95))
scaler = torch.cuda.amp.GradScaler(enabled=(device == 'cuda')) # For Mixed Precision

def get_batch(split):
    filename = 'train.bin' if split == 'train' else 'val.bin'
    data = np.memmap(filename, dtype=np.uint16, mode='r')
    ix = torch.randint(len(data) - config.block_size, (batch_size,))
    x = torch.stack([torch.from_numpy((data[i:i+config.block_size]).astype(np.int64)) for i in ix])
    y = torch.stack([torch.from_numpy((data[i+1:i+1+config.block_size]).astype(np.int64)) for i in ix])
    if device == 'cuda':
        x, y = x.pin_memory().to(device, non_blocking=True), y.pin_memory().to(device, non_blocking=True)
    return x, y

@torch.no_grad()
def estimate_loss():
    out = {}
    model.eval()
    for split in ['train']:
        losses = torch.zeros(100) # smaller eval for speed
        for k in range(100):
            X, Y = get_batch(split)
            with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
                logits, loss = model(X, Y)
            losses[k] = loss.item()
        out[split] = losses.mean()
    model.train()
    return out

# Scheduler
def get_lr(it):
    if it < warmup_iters:
        return learning_rate * it / warmup_iters
    if it > max_iters:
        return learning_rate / 10
    decay_ratio = (it - warmup_iters) / (max_iters - warmup_iters)
    coeff = 0.5 * (1.0 + math.cos(math.pi * decay_ratio))
    return learning_rate * 0.1 + coeff * (learning_rate * 0.9)

# Training
t0 = time.time()
print("Starting training...")

for iter in range(max_iters):
    # Set learning rate
    lr = get_lr(iter)
    for param_group in optimizer.param_groups:
        param_group['lr'] = lr

    # Forward backward update
    for micro_step in range(gradient_accumulation_steps):
        X, Y = get_batch('train')
        with torch.amp.autocast(device_type="cuda", dtype=torch.float16):
            logits, loss = model(X, Y)
            loss = loss / gradient_accumulation_steps # scale the loss to account for gradient accumulation
        scaler.scale(loss).backward()

    scaler.step(optimizer)
    scaler.update()
    optimizer.zero_grad(set_to_none=True)

    # Logging
    if iter % 100 == 0:
        dt = time.time() - t0
        t0 = time.time()
        print(f"Step {iter}: Loss {loss.item() * gradient_accumulation_steps:.4f} | Time: {dt*1000:.2f}ms")

    if iter % 1000 == 0 and iter > 0:
        losses = estimate_loss()
        print(f"--- Eval Step {iter}: Train Loss {losses['train']:.4f} ---")

print("Training Complete!")
torch.save(model.state_dict(), "model_50m.pt")

In [ ]:
print("\nGenerating Text...")
model.eval()
context = torch.zeros((1, 1), dtype=torch.long, device=device) # Start with token 0
generated = model.generate(context, max_new_tokens=100)
print(enc.decode(generated[0].tolist()))

In [ ]:
# -----------------------------------------------------------------------------
# FIXED EXPORT SCRIPT (Handles '_orig_mod' prefix from torch.compile)
# -----------------------------------------------------------------------------
import struct
import os
import torch
import numpy as np
from google.colab import drive
import shutil
import tiktoken

# 1. Mount Drive
if not os.path.exists('/content/drive'):
    print("Mounting Google Drive...")
    drive.mount('/content/drive')

drive_folder = "/content/drive/MyDrive/My_SLM_Project"
os.makedirs(drive_folder, exist_ok=True)

# 2. Load Model & Fix Keys
print("Exporting Model...")
device = 'cpu'
model = GPT(config) # Raw model structure

# Load state dict
if os.path.exists("model_50m.pt"):
    state_dict = torch.load("model_50m.pt", map_location=device)

    # --- CRITICAL FIX START ---
    # Loop through keys and remove '_orig_mod.' prefix
    unwanted_prefix = '_orig_mod.'
    for k, v in list(state_dict.items()):
        if k.startswith(unwanted_prefix):
            new_key = k[len(unwanted_prefix):]
            state_dict[new_key] = state_dict.pop(k)
    # --- CRITICAL FIX END ---

    model.load_state_dict(state_dict)
    print("Model loaded successfully (Prefixes removed).")
else:
    print("Error: model_50m.pt not found!")

model.eval()

# Helper to write tensors (FIXED for bias=False)
def serialize(t, shape_hint=None):
    if t is None:
        # Create dummy zeros matching the expected shape to keep C++ happy
        d = np.zeros(shape_hint, dtype=np.float32)
    else:
        d = t.detach().cpu().view(-1).numpy().astype(np.float32)
    b = struct.pack(f'{len(d)}f', *d)
    return b

with open("model.bin", "wb") as f:
    # Header
    header = struct.pack('iiiii', config.n_layer, config.n_head, config.n_embd, config.block_size, config.vocab_size)
    f.write(header)

    # Writes
    f.write(serialize(model.transformer.wte.weight))
    f.write(serialize(model.transformer.wpe.weight))

    for block in model.transformer.h:
        f.write(serialize(block.ln1.weight))
        f.write(serialize(block.ln1.bias, shape_hint=config.n_embd))

        f.write(serialize(block.attn.c_attn.weight))
        f.write(serialize(block.attn.c_attn.bias, shape_hint=3*config.n_embd))

        f.write(serialize(block.attn.c_proj.weight))
        f.write(serialize(block.attn.c_proj.bias, shape_hint=config.n_embd))

        f.write(serialize(block.ln2.weight))
        f.write(serialize(block.ln2.bias, shape_hint=config.n_embd))

        f.write(serialize(block.mlp.c_fc.weight))
        f.write(serialize(block.mlp.c_fc.bias, shape_hint=4*config.n_embd))
        f.write(serialize(block.mlp.c_proj.weight))
        f.write(serialize(block.mlp.c_proj.bias, shape_hint=config.n_embd))

    f.write(serialize(model.transformer.ln_f.weight))
    f.write(serialize(model.transformer.ln_f.bias, shape_hint=config.n_embd))
    f.write(serialize(model.lm_head.weight))

print("-> model.bin created.")

# 3. Tokenizer Export
print("Exporting Tokenizer...")
enc = tiktoken.get_encoding("gpt2")
with open("tokenizer.bin", "wb") as f:
    vocab_size = 50257
    f.write(struct.pack('i', vocab_size))
    for i in range(vocab_size):
        try:
            b = enc.decode_bytes([i])
        except:
            b = b"<ERR>"
        f.write(struct.pack('i', len(b)))
        f.write(b)
print("-> tokenizer.bin created.")

# 4. Save to Drive
print(f"Saving to {drive_folder}...")
shutil.copy("model.bin", os.path.join(drive_folder, "model.bin"))
shutil.copy("tokenizer.bin", os.path.join(drive_folder, "tokenizer.bin"))
print("DONE! Files safely saved in Drive.")

In [ ]:
# -----------------------------------------------------------------------------
# FIXED TESTING CELL (Handles '_orig_mod' prefix automatically)
# -----------------------------------------------------------------------------
import torch
import textwrap
import os

# 1. Setup
device = 'cuda' if torch.cuda.is_available() else 'cpu'
print(f"Testing on: {device}")

# 2. Load the BEST saved model with Prefix Fix
model = GPT(config) # Create raw model

if os.path.exists("model_50m.pt"):
    print("Loading model weights...")
    state_dict = torch.load("model_50m.pt", map_location=device)

    # --- THE FIX: Clean the keys ---
    unwanted_prefix = '_orig_mod.'
    for k, v in list(state_dict.items()):
        if k.startswith(unwanted_prefix):
            new_key = k[len(unwanted_prefix):]
            state_dict[new_key] = state_dict.pop(k)
    # -------------------------------

    model.load_state_dict(state_dict)
    print("Success! Weights loaded and prefixes cleaned.")
else:
    print("Warning: 'model_50m.pt' not found. Using random weights.")

model.eval()
model.to(device)

def generate_story(prompt_text, max_length=250, temperature=0.8):
    print(f"\n{'='*50}")
    print(f"PROMPT: {prompt_text}")
    print(f"{'='*50}")

    # Encode prompt
    try:
        input_ids = enc.encode(prompt_text)
    except AttributeError:
        # Fallback if enc is simple
        input_ids = enc.encode_ordinary(prompt_text)

    context = torch.tensor(input_ids, dtype=torch.long, device=device).unsqueeze(0)

    # Generate
    with torch.no_grad():
        output_ids = model.generate(context, max_new_tokens=max_length, temperature=temperature)

    # Decode
    generated_text = enc.decode(output_ids[0].tolist())

    # Format output for readability
    print("\nGENERATED STORY:\n")
    print(textwrap.fill(generated_text, width=80))
    print("-" * 50)

# --- TEST CASES ---
generate_story("Once upon a time, there was a little", max_length=200, temperature=0.8)
generate_story("The big dog was very angry because", max_length=200, temperature=0.9)

In [ ]:
# ============================================================================
# COPY THIS ENTIRE CELL TO YOUR COLAB NOTEBOOK
# Run this AFTER training your model (after the training loop)
# ============================================================================

!pip install -q psutil  # For memory monitoring

import time
import numpy as np
import torch
import psutil
import os
from tqdm.auto import tqdm

# ---------------------------------------------------------------------------
# Quick Benchmark Function (Lightweight for Colab)
# ---------------------------------------------------------------------------

def quick_benchmark(model, enc, device='cuda', num_runs=20):
    """
    Fast benchmark - measures key metrics without heavy dependencies
    """
    print("\n" + "="*70)
    print("  SLM PERFORMANCE BENCHMARK")
    print("="*70 + "\n")

    # System info
    print("📊 SYSTEM:")
    print(f"   Device: {device.upper()}")
    if torch.cuda.is_available():
        print(f"   GPU: {torch.cuda.get_device_name(0)}")
        print(f"   GPU Memory: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
    print()

    # Model info
    total_params = sum(p.numel() for p in model.parameters())
    print("🤖 MODEL:")
    print(f"   Parameters: {total_params/1e6:.2f}M")
    print(f"   Layers: {config.n_layer} | Heads: {config.n_head} | Dim: {config.n_embd}")
    print()

    # Prepare
    model.eval()
    model.to(device)

    test_prompts = [
        "Once upon a time",
        "The big dog was",
        "In a galaxy far",
        "There was a little",
        "The brave knight"
    ]

    # Warmup (important for accurate timing)
    print("🔥 Warming up (3 runs)...")
    with torch.no_grad():
        for _ in range(3):
            input_ids = enc.encode(test_prompts[0])
            context = torch.tensor(input_ids, dtype=torch.long, device=device).unsqueeze(0)
            _ = model.generate(context, max_new_tokens=50, temperature=0.8)

    # Benchmark
    print(f"⚡ Running {num_runs} inference passes...\n")

    latencies = []
    tokens_generated = []

    # Memory before
    if device == 'cuda':
        torch.cuda.reset_peak_memory_stats()
        mem_before = torch.cuda.memory_allocated() / 1024**2
    else:
        process = psutil.Process(os.getpid())
        mem_before = process.memory_info().rss / 1024**2

    with torch.no_grad():
        for i in tqdm(range(num_runs), desc="Benchmarking"):
            prompt = test_prompts[i % len(test_prompts)]
            input_ids = enc.encode(prompt)
            context = torch.tensor(input_ids, dtype=torch.long, device=device).unsqueeze(0)

            # Time the generation
            start = time.perf_counter()
            output = model.generate(context, max_new_tokens=50, temperature=0.8, top_k=40)
            end = time.perf_counter()

            latency_ms = (end - start) * 1000
            latencies.append(latency_ms)
            tokens_generated.append(50)  # We asked for 50 tokens

    # Memory after
    if device == 'cuda':
        mem_after = torch.cuda.memory_allocated() / 1024**2
        mem_peak = torch.cuda.max_memory_allocated() / 1024**2
    else:
        mem_after = process.memory_info().rss / 1024**2
        mem_peak = mem_after

    # Calculate metrics
    avg_latency = np.mean(latencies)
    std_latency = np.std(latencies)
    min_latency = np.min(latencies)
    max_latency = np.max(latencies)

    avg_tokens = np.mean(tokens_generated)
    total_tokens = sum(tokens_generated)
    total_time = sum(latencies) / 1000  # seconds

    latency_per_token = avg_latency / avg_tokens
    throughput = total_tokens / total_time

    # Print results
    print("\n" + "="*70)
    print("  RESULTS")
    print("="*70 + "\n")

    print("⏱️  LATENCY:")
    print(f"   Total (50 tokens):  {avg_latency:.2f} ms  (± {std_latency:.2f} ms)")
    print(f"   Per Token:          {latency_per_token:.2f} ms/token")
    print(f"   Range:              {min_latency:.2f} - {max_latency:.2f} ms")
    print()

    print("🚀 THROUGHPUT:")
    print(f"   Speed:              {throughput:.1f} tokens/second")
    print(f"   Total Tokens:       {total_tokens} in {total_time:.2f}s")
    print()

    print("💾 MEMORY:")
    print(f"   Used:               {mem_after:.1f} MB")
    print(f"   Peak:               {mem_peak:.1f} MB")
    print(f"   Delta:              {mem_after - mem_before:.1f} MB")
    print()

    # Quality test - generate 5 samples and check diversity
    print("🎨 QUALITY CHECK (generating 5 samples):")
    samples = []
    with torch.no_grad():
        for i in range(5):
            input_ids = enc.encode("Once upon a time, there was a")
            context = torch.tensor(input_ids, dtype=torch.long, device=device).unsqueeze(0)
            output = model.generate(context, max_new_tokens=30, temperature=0.9)
            text = enc.decode(output[0].tolist())
            samples.append(text)
            print(f"\n   {i+1}. {text[:100]}...")

    unique = len(set(samples))
    print(f"\n   Unique outputs: {unique}/5 ({unique/5*100:.0f}% diversity)")

    print("\n" + "="*70)
    print("  SUMMARY FOR RESUME/GITHUB")
    print("="*70)
    print(f"""
✓ Model: {total_params/1e6:.1f}M parameters GPT-2 architecture
✓ Speed: {throughput:.1f} tokens/sec ({latency_per_token:.2f} ms/token)
✓ Memory: {mem_peak:.0f} MB peak usage
✓ Quality: Coherent stories with {unique/5*100:.0f}% output diversity
""")

    return {
        'throughput': throughput,
        'latency_per_token': latency_per_token,
        'avg_latency': avg_latency,
        'memory_mb': mem_peak,
        'diversity': unique/5*100
    }

# ---------------------------------------------------------------------------
# RUN THE BENCHMARK
# ---------------------------------------------------------------------------

# Make sure 'model', 'config', and 'enc' are defined from your training code
import tiktoken
enc = tiktoken.get_encoding("gpt2")

# Run it!
results = quick_benchmark(
    model=model,
    enc=enc,
    device=device,  # Should be 'cuda' or 'cpu'
    num_runs=20     # Increase to 50 for more accurate results
)

# Optional: Save results
print("\n💾 Saving benchmark results...")
import json
with open('benchmark_results.json', 'w') as f:
    json.dump(results, f, indent=2)
print("   Saved to: benchmark_results.json")

In [ ]:
"""
MINIMAL BENCHMARK - PASTE THIS CELL IN COLAB AFTER TRAINING
Just copy and run - no extra files needed!
"""

import time, numpy as np, torch
from tqdm.auto import tqdm

print("\n🔥 Starting SLM Benchmark...\n")

# Warmup
model.eval()
with torch.no_grad():
    for _ in range(3):
        ctx = torch.tensor(enc.encode("Once upon a time"), device=device).unsqueeze(0)
        _ = model.generate(ctx, max_new_tokens=50, temperature=0.8)

# Benchmark
prompts = ["Once upon a time", "The big dog", "In space", "There was", "A brave knight"]
latencies = []

print("Running 20 inference tests...")
with torch.no_grad():
    for i in tqdm(range(20)):
        ctx = torch.tensor(enc.encode(prompts[i % 5]), device=device).unsqueeze(0)

        start = time.perf_counter()
        out = model.generate(ctx, max_new_tokens=50, temperature=0.8, top_k=40)
        latency = (time.perf_counter() - start) * 1000

        latencies.append(latency)

# Results
avg = np.mean(latencies)
per_token = avg / 50
throughput = 50 / (avg / 1000)
params = sum(p.numel() for p in model.parameters()) / 1e6

print(f"""
{'='*60}
📊 BENCHMARK RESULTS
{'='*60}

Model:           {params:.1f}M parameters
Total Latency:   {avg:.2f} ms (for 50 tokens)
Per Token:       {per_token:.2f} ms/token
Throughput:      {throughput:.1f} tokens/second

{'='*60}
✅ COPY THIS TO YOUR RESUME/GITHUB:
   • {params:.1f}M parameter transformer
   • {throughput:.0f} tokens/sec inference speed
   • {per_token:.1f}ms latency per token
{'='*60}
""")

# Generate a sample
print("\n📝 Sample Generation:")
with torch.no_grad():
    ctx = torch.tensor(enc.encode("Once upon a time, there was a"), device=device).unsqueeze(0)
    out = model.generate(ctx, max_new_tokens=80, temperature=0.9)
    print(enc.decode(out[0].tolist()))

In [ ]:
"""
benchmark_slm.py - Complete Benchmarking Suite for SLM
Run this in Google Colab after training your model

Measures:
1. Inference latency (ms per token)
2. Throughput (tokens/second)
3. Memory usage (MB)
4. C++ vs Python speed comparison
"""

import time
import numpy as np
import torch
import psutil
import os
from tqdm.auto import tqdm

# ---------------------------------------------------------------------------
# Helper Functions
# ---------------------------------------------------------------------------

def get_memory_mb():
    """Get current process memory usage in MB"""
    process = psutil.Process(os.getpid())
    return process.memory_info().rss / 1024 / 1024

def get_gpu_memory_mb():
    """Get GPU memory if available"""
    if torch.cuda.is_available():
        return torch.cuda.memory_allocated() / 1024 / 1024
    return 0

# ---------------------------------------------------------------------------
# Benchmark 1: Python (PyTorch) Inference
# ---------------------------------------------------------------------------

def benchmark_python_inference(model, enc, device='cuda', num_runs=10):
    """Benchmark the PyTorch model directly"""
    print("\n" + "="*60)
    print("BENCHMARK 1: PyTorch Model (Python)")
    print("="*60)

    model.eval()
    model.to(device)

    prompts = [
        "Once upon a time",
        "The big dog",
        "In a galaxy far away",
        "There was a little girl",
        "The spaceship landed"
    ]

    latencies = []
    token_counts = []

    # Warmup
    print("Warming up...")
    with torch.no_grad():
        for _ in range(3):
            prompt = prompts[0]
            input_ids = enc.encode(prompt)
            context = torch.tensor(input_ids, dtype=torch.long, device=device).unsqueeze(0)
            _ = model.generate(context, max_new_tokens=50, temperature=0.8)

    # Actual benchmark
    print(f"Running {num_runs} inference passes...")
    mem_before = get_memory_mb()
    gpu_mem_before = get_gpu_memory_mb()

    with torch.no_grad():
        for i in tqdm(range(num_runs)):
            prompt = prompts[i % len(prompts)]
            input_ids = enc.encode(prompt)
            context = torch.tensor(input_ids, dtype=torch.long, device=device).unsqueeze(0)

            start = time.perf_counter()
            output = model.generate(context, max_new_tokens=50, temperature=0.8)
            latency = (time.perf_counter() - start) * 1000  # Convert to ms

            latencies.append(latency)
            token_counts.append(50)  # We generated 50 tokens

    mem_after = get_memory_mb()
    gpu_mem_after = get_gpu_memory_mb()

    # Results
    avg_latency = np.mean(latencies)
    std_latency = np.std(latencies)
    avg_tokens = np.mean(token_counts)
    throughput = (avg_tokens / (avg_latency / 1000))  # tokens/second

    print("\n--- PYTHON RESULTS ---")
    print(f"Average Latency:     {avg_latency:.2f} ± {std_latency:.2f} ms")
    print(f"Latency per Token:   {avg_latency/avg_tokens:.2f} ms/token")
    print(f"Throughput:          {throughput:.1f} tokens/second")
    print(f"Memory Usage:        {mem_after - mem_before:.1f} MB delta")
    if device == 'cuda':
        print(f"GPU Memory:          {gpu_mem_after:.1f} MB")

    return {
        'avg_latency': avg_latency,
        'throughput': throughput,
        'memory_mb': mem_after - mem_before
    }

# ---------------------------------------------------------------------------
# Benchmark 2: C++ Inference (if compiled)
# ---------------------------------------------------------------------------

def benchmark_cpp_inference(lib, enc, model_path="model.bin", num_runs=10):
    """Benchmark the C++ inference engine"""
    print("\n" + "="*60)
    print("BENCHMARK 2: C++ Inference Engine")
    print("="*60)

    import ctypes

    # Initialize
    print("Initializing C++ model...")
    lib.init_model(model_path.encode("utf-8"))

    prompts = [
        "Once upon a time",
        "The big dog",
        "In a galaxy far away",
        "There was a little girl",
        "The spaceship landed"
    ]

    latencies = []
    token_counts = []

    # Warmup
    print("Warming up...")
    for _ in range(3):
        input_ids = enc.encode(prompts[0])
        InputArray = ctypes.c_int * len(input_ids)
        OutputArray = ctypes.c_int * 50
        input_c = InputArray(*input_ids)
        output_c = OutputArray()

        _ = lib.generate(
            input_c,
            ctypes.c_int(len(input_ids)),
            output_c,
            ctypes.c_int(50),
            ctypes.c_float(0.8),
            ctypes.c_int(40)
        )

    # Actual benchmark
    print(f"Running {num_runs} inference passes...")
    mem_before = get_memory_mb()

    for i in tqdm(range(num_runs)):
        prompt = prompts[i % len(prompts)]
        input_ids = enc.encode(prompt)

        InputArray = ctypes.c_int * len(input_ids)
        OutputArray = ctypes.c_int * 50
        input_c = InputArray(*input_ids)
        output_c = OutputArray()

        start = time.perf_counter()
        n_generated = lib.generate(
            input_c,
            ctypes.c_int(len(input_ids)),
            output_c,
            ctypes.c_int(50),
            ctypes.c_float(0.8),
            ctypes.c_int(40)
        )
        latency = (time.perf_counter() - start) * 1000

        latencies.append(latency)
        token_counts.append(n_generated)

    mem_after = get_memory_mb()

    # Results
    avg_latency = np.mean(latencies)
    std_latency = np.std(latencies)
    avg_tokens = np.mean(token_counts)
    throughput = (avg_tokens / (avg_latency / 1000))

    print("\n--- C++ RESULTS ---")
    print(f"Average Latency:     {avg_latency:.2f} ± {std_latency:.2f} ms")
    print(f"Latency per Token:   {avg_latency/avg_tokens:.2f} ms/token")
    print(f"Throughput:          {throughput:.1f} tokens/second")
    print(f"Memory Usage:        {mem_after - mem_before:.1f} MB delta")

    lib.cleanup_model()

    return {
        'avg_latency': avg_latency,
        'throughput': throughput,
        'memory_mb': mem_after - mem_before
    }

# ---------------------------------------------------------------------------
# Benchmark 3: Model Quality Metrics
# ---------------------------------------------------------------------------

def benchmark_generation_quality(model, enc, device='cuda', num_samples=20):
    """Test generation quality and diversity"""
    print("\n" + "="*60)
    print("BENCHMARK 3: Generation Quality")
    print("="*60)

    model.eval()
    model.to(device)

    prompts = [
        "Once upon a time, there was a",
        "The brave knight",
        "In the deep forest",
    ]

    all_outputs = []
    unique_outputs = set()

    print(f"Generating {num_samples} samples...")
    with torch.no_grad():
        for i in tqdm(range(num_samples)):
            prompt = prompts[i % len(prompts)]
            input_ids = enc.encode(prompt)
            context = torch.tensor(input_ids, dtype=torch.long, device=device).unsqueeze(0)

            output = model.generate(context, max_new_tokens=30, temperature=0.9)
            text = enc.decode(output[0].tolist())
            all_outputs.append(text)
            unique_outputs.add(text)

    # Calculate diversity
    diversity = len(unique_outputs) / len(all_outputs) * 100

    print("\n--- QUALITY RESULTS ---")
    print(f"Total Samples:       {len(all_outputs)}")
    print(f"Unique Outputs:      {len(unique_outputs)}")
    print(f"Diversity Score:     {diversity:.1f}%")
    print(f"\nSample Outputs:")
    for i, text in enumerate(all_outputs[:3]):
        print(f"\n{i+1}. {text[:150]}...")

    return {
        'diversity': diversity,
        'num_unique': len(unique_outputs)
    }

# ---------------------------------------------------------------------------
# Main Benchmark Runner
# ---------------------------------------------------------------------------

def run_full_benchmark(model, enc, config, device='cuda', include_cpp=False):
    """Run complete benchmark suite"""

    print("\n" + "╔" + "="*58 + "╗")
    print("║" + " "*15 + "SLM BENCHMARK SUITE" + " "*24 + "║")
    print("╚" + "="*58 + "╝\n")

    # System Info
    print("SYSTEM INFORMATION:")
    print(f"  Device:            {device}")
    print(f"  CPU Cores:         {psutil.cpu_count()}")
    print(f"  RAM Available:     {psutil.virtual_memory().available / 1024**3:.1f} GB")
    if torch.cuda.is_available():
        print(f"  GPU:               {torch.cuda.get_device_name(0)}")
        print(f"  GPU Memory:        {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

    # Model Info
    print("\nMODEL CONFIGURATION:")
    print(f"  Layers:            {config.n_layer}")
    print(f"  Heads:             {config.n_head}")
    print(f"  Embedding Dim:     {config.n_embd}")
    print(f"  Context Length:    {config.block_size}")
    print(f"  Vocab Size:        {config.vocab_size}")

    # Calculate params
    total_params = sum(p.numel() for p in model.parameters())
    print(f"  Total Parameters:  {total_params/1e6:.2f}M")

    # Run benchmarks
    results = {}

    # Python benchmark
    results['python'] = benchmark_python_inference(model, enc, device, num_runs=20)

    # Quality benchmark
    results['quality'] = benchmark_generation_quality(model, enc, device, num_samples=20)

    # C++ benchmark (optional - if compiled)
    if include_cpp:
        try:
            import ctypes
            lib = ctypes.CDLL("./libllm.so")
            # Define function signatures (simplified - add full definitions)
            lib.init_model.argtypes = [ctypes.c_char_p]
            lib.cleanup_model.argtypes = []

            results['cpp'] = benchmark_cpp_inference(lib, enc, num_runs=20)

            # Comparison
            print("\n" + "="*60)
            print("PYTHON vs C++ COMPARISON")
            print("="*60)
            speedup = results['python']['avg_latency'] / results['cpp']['avg_latency']
            print(f"Speedup Factor:      {speedup:.2f}x")
            print(f"Latency Reduction:   {(1 - 1/speedup)*100:.1f}%")

        except Exception as e:
            print(f"\nC++ benchmark skipped: {e}")

    # Final Summary
    print("\n" + "╔" + "="*58 + "╗")
    print("║" + " "*20 + "FINAL SUMMARY" + " "*25 + "║")
    print("╚" + "="*58 + "╝\n")

    print(f"✓ Inference Speed:   {results['python']['throughput']:.1f} tokens/second")
    print(f"✓ Latency per Token: {results['python']['avg_latency']/50:.2f} ms")
    print(f"✓ Generation Diversity: {results['quality']['diversity']:.1f}%")
    print(f"✓ Memory Efficient:  ~{results['python']['memory_mb']:.0f} MB overhead")

    if 'cpp' in results:
        print(f"✓ C++ Speedup:       {results['python']['avg_latency'] / results['cpp']['avg_latency']:.2f}x faster")

    return results


# ---------------------------------------------------------------------------
# USAGE EXAMPLE (Add to your Colab notebook)
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    """
    Copy this block into your Colab notebook AFTER training:

    # Run benchmark
    import tiktoken
    enc = tiktoken.get_encoding("gpt2")

    results = run_full_benchmark(
        model=model,           # Your trained GPT model
        enc=enc,              # Tokenizer
        config=config,        # Your GPTConfig
        device='cuda',        # or 'cpu'
        include_cpp=False     # Set True if you compiled C++ version
    )
    """
    print("Import this file in Colab and call run_full_benchmark()")